# CAN TF LoRA (Jupyter)

Run all steps from this notebook to fine-tune LoRA for `mistralai/Ministral-3-8B-Instruct-2512` using Hugging Face PEFT/QLoRA.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'torch'])

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'transformers', 'accelerate'])

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'peft'])

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'datasets', 'trl', 'tqdm'])

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'bitsandbytes'])

In [ ]:
import peft, transformers, torch
print('peft:', peft.__version__)
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)


In [ ]:
from pathlib import Path
import sys

# Edit these values as needed
MODEL_ID = "mistralai/Ministral-3-8B-Instruct-2512"
SELECTED_DATASETS = ["DoS"]  # e.g. ["DoS", "Fuzzy", "Gear", "RPM"]
MAX_TRAIN_SAMPLES = 512
MAX_EVAL_SAMPLES = 128
EPOCHS = 1.0
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
USE_4BIT = True
TRAIN_OUTPUT_DIR = "lora_tf_hf_ministral3"

SCRIPT_PATH = Path('Answer_QA_TF_LLM_LoRA.py').resolve()
if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Missing script: {SCRIPT_PATH}")

print('Using script:', SCRIPT_PATH)
print('Python:', sys.executable)

In [ ]:
import subprocess
import shlex

cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--model_id', MODEL_ID,
    '--selected_datasets', *SELECTED_DATASETS,
    '--max_train_samples', str(MAX_TRAIN_SAMPLES),
    '--max_eval_samples', str(MAX_EVAL_SAMPLES),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--grad_accum_steps', str(GRAD_ACCUM_STEPS),
    '--train_output_dir', TRAIN_OUTPUT_DIR,
]

if USE_4BIT:
    cmd.append('--use_4bit')
else:
    cmd.append('--no-use_4bit')

print('Running command:')
print(' '.join(shlex.quote(c) for c in cmd))

result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f'LoRA run failed with exit code {result.returncode}')

print('LoRA run completed successfully.')

In [ ]:
from pathlib import Path

out_dir = Path(TRAIN_OUTPUT_DIR)
adapter_dir = out_dir / 'adapter'
print('Output dir exists:', out_dir.exists(), out_dir)
print('Adapter dir exists:', adapter_dir.exists(), adapter_dir)
if adapter_dir.exists():
    print('Adapter files:')
    for p in sorted(adapter_dir.glob('*')):
        print(' -', p.name)